# Notebook 1 — Official Baseline Reconstruction + Model Selection
This notebook does not run quantum simulations. It starts from variance_curves.csv, derives the baseline \(\tau_{BP}\) values from the stored var_pooled curves, and then independently discovers which scaling model is best supported.
It deliberately does not hardcode \(A\), \(c\), \(R^2\), the model winner, or any expected result.
Run all cells in order in a fresh Colab runtime.

## Cell 1 — Imports and configuration

In [ ]:
# ============================================================
# NOTEBOOK 1 — OFFICIAL BASELINE RECONSTRUCTION
# ============================================================
#
# Scientific purpose:
# Reconstruct the baseline tau_BP dataset from the stored
# depth-by-depth gradient variance measurements and determine
# which candidate scaling model is best supported by the data.
#
# IMPORTANT:
# - NO quantum simulations are run.
# - NO previous fitted A/c values are used.
# - NO model winner is hardcoded.
# - Raw variance_curves.csv is never modified.
#
# Primary fitting domain:
#     n = 8, 10, 12
#
# Held-out data:
#     n = 14, 16
#
# These held-out systems are NOT used in fitting.
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

INPUT_FILE = "../data/variance_curves.csv"

BASELINE_THRESHOLD = 1e-2

TRAIN_N = [8, 10, 12]
HELD_OUT_N = [14, 16]

OUTPUT_DIR = "notebook1_baseline"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 80)
print("NOTEBOOK 1 — OFFICIAL BASELINE RECONSTRUCTION")
print("=" * 80)
print(f"Input file:          {INPUT_FILE}")
print(f"Baseline threshold:  {BASELINE_THRESHOLD}")
print(f"Training n values:   {TRAIN_N}")
print(f"Held-out n values:   {HELD_OUT_N}")
print("Quantum simulation:  NONE")
print("=" * 80)

NOTEBOOK 1 — OFFICIAL BASELINE RECONSTRUCTION
Input file:          variance_curves.csv
Baseline threshold:  0.01
Training n values:   [8, 10, 12]
Held-out n values:   [14, 16]
Quantum simulation:  NONE


## Cell 2 — Load and validate the raw variance data

In [2]:
# ============================================================
# CELL 2 — LOAD RAW VARIANCE CURVES
# ============================================================

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"CRITICAL ERROR: '{INPUT_FILE}' was not found.\n"
        "Place the raw variance_curves.csv file in the Colab working directory."
    )

raw_df = pd.read_csv(INPUT_FILE)

required_columns = [
    "n",
    "k",
    "depth",
    "var_pooled",
]

missing = [c for c in required_columns if c not in raw_df.columns]

if missing:
    raise ValueError(
        f"CRITICAL ERROR: Missing required columns: {missing}"
    )

# Convert required fields to numeric
for col in ["n", "k", "depth", "var_pooled"]:
    raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

if raw_df[required_columns].isna().any().any():
    bad_counts = raw_df[required_columns].isna().sum()
    raise ValueError(
        "CRITICAL ERROR: Missing/non-numeric values detected:\n"
        f"{bad_counts[bad_counts > 0]}"
    )

print("\nRAW DATASET")
print("-" * 80)
print(f"Rows:        {len(raw_df):,}")
print(f"n values:    {sorted(raw_df['n'].unique().astype(int).tolist())}")
print(f"k values:    {sorted(raw_df['k'].unique().astype(int).tolist())}")
print(
    f"Depth range: {int(raw_df['depth'].min())} "
    f"to {int(raw_df['depth'].max())}"
)

if "architecture" in raw_df.columns:
    print(
        "Architecture(s):",
        raw_df["architecture"].dropna().unique().tolist()
    )

if "framework" in raw_df.columns:
    print(
        "Framework(s):",
        raw_df["framework"].dropna().unique().tolist()
    )

if "n_param_sets" in raw_df.columns:
    print(
        "Parameter-set counts:",
        sorted(raw_df["n_param_sets"].dropna().unique().tolist())
    )

FileNotFoundError: CRITICAL ERROR: 'variance_curves.csv' was not found.
Place the raw variance_curves.csv file in the Colab working directory.

## Cell 3 — Critical methodology audit

In [ ]:
# ============================================================
# CELL 3 — METHODOLOGY AUDIT
# ============================================================

print("\n" + "=" * 80)
print("METHODOLOGY AUDIT")
print("=" * 80)

audit_pass = True

# ------------------------------------------------------------
# 1. Architecture consistency
# ------------------------------------------------------------

if "architecture" in raw_df.columns:
    arch_counts = raw_df.groupby(["n", "k"])["architecture"].nunique()

    bad_arch = arch_counts[arch_counts > 1]

    if len(bad_arch) > 0:
        print("FAIL: Multiple architectures found for some (n,k) configurations:")
        print(bad_arch)
        audit_pass = False
    else:
        print("PASS: One architecture per (n,k) configuration.")

# ------------------------------------------------------------
# 2. k <= n
# ------------------------------------------------------------

invalid_k = raw_df[raw_df["k"] > raw_df["n"]]

if len(invalid_k) > 0:
    print(f"FAIL: Found {len(invalid_k)} rows where k > n.")
    audit_pass = False
else:
    print("PASS: All rows satisfy k <= n.")

# ------------------------------------------------------------
# 3. Non-negative variance
# ------------------------------------------------------------

negative_variance = raw_df[raw_df["var_pooled"] < 0]

if len(negative_variance) > 0:
    print(f"FAIL: Found {len(negative_variance)} negative variance values.")
    audit_pass = False
else:
    print("PASS: All pooled variances are non-negative.")

# ------------------------------------------------------------
# 4. Duplicate (n,k,depth) records
# ------------------------------------------------------------

duplicate_keys = (
    raw_df.groupby(["n", "k", "depth"])
    .size()
    .reset_index(name="count")
)

duplicate_keys = duplicate_keys[duplicate_keys["count"] > 1]

if len(duplicate_keys) > 0:
    print(
        "WARNING: Multiple rows exist for some (n,k,depth) keys."
    )
    print(duplicate_keys.head(20))
    print(
        "These must be reviewed before treating each row as a unique "
        "depth measurement."
    )
else:
    print("PASS: Exactly one row per (n,k,depth).")

# ------------------------------------------------------------
# 5. Architecture count
# ------------------------------------------------------------

if "architecture" in raw_df.columns:
    unique_arch = raw_df["architecture"].dropna().unique()

    if len(unique_arch) != 1:
        print(
            "WARNING: More than one architecture exists in the raw file:",
            unique_arch.tolist()
        )
    else:
        print(
            f"PASS: Single architecture in baseline raw file: {unique_arch[0]}"
        )

# ------------------------------------------------------------
# 6. Parameter count consistency, if available
# ------------------------------------------------------------

if "n_params" in raw_df.columns:
    expected = raw_df["depth"] * raw_df["n"]

    mismatches = ~np.isclose(
        raw_df["n_params"].astype(float),
        expected.astype(float),
        rtol=0,
        atol=0
    )

    if mismatches.any():
        print(
            f"WARNING: {int(mismatches.sum())} rows have "
            "n_params != depth*n."
        )
    else:
        print("PASS: n_params = depth*n throughout.")

print()

if audit_pass:
    print("OVERALL AUDIT STATUS: PASS")
else:
    print("OVERALL AUDIT STATUS: FAIL")
    raise RuntimeError(
        "Critical raw-data audit failure. Stop before fitting."
    )

## Cell 4 — Check depth coverage

In [ ]:
# ============================================================
# CELL 4 — DEPTH COVERAGE AUDIT
# ============================================================

depth_audit = []

for (n, k), group in raw_df.groupby(["n", "k"]):

    depths = sorted(group["depth"].astype(int).unique())

    expected = set(range(min(depths), max(depths) + 1))
    actual = set(depths)

    missing_depths = sorted(expected - actual)

    depth_audit.append({
        "n": int(n),
        "k": int(k),
        "min_depth": int(min(depths)),
        "max_depth": int(max(depths)),
        "n_depths": len(depths),
        "missing_internal_depths": missing_depths,
        "complete_contiguous": len(missing_depths) == 0,
    })

depth_audit_df = pd.DataFrame(depth_audit)

print("=" * 80)
print("DEPTH COVERAGE AUDIT")
print("=" * 80)

display(depth_audit_df)

if not depth_audit_df["complete_contiguous"].all():
    print(
        "WARNING: Some configurations have missing internal depths. "
        "Review before interpreting tau."
    )
else:
    print("PASS: All configurations have contiguous depth coverage.")

## Cell 5 — Reconstruct \(\tau_{BP}\)

In [ ]:
# ============================================================
# CELL 5 — RECONSTRUCT TAU_BP
# ============================================================
#
# Definition:
#
#   tau_BP = FIRST depth L where:
#
#       var_pooled(L) < 1e-2
#
# If the threshold is never crossed:
#
#       tau_BP = MAX_DEPTH + 1
#
#       censored = True
#
# This cell does NOT fit any model.
# ============================================================

MAX_DEPTH = int(raw_df["depth"].max())

tau_records = []

for (n, k), group in raw_df.groupby(["n", "k"]):

    group = group.sort_values("depth")

    crossed = group[
        group["var_pooled"] < BASELINE_THRESHOLD
    ]

    if len(crossed) > 0:
        tau = int(crossed.iloc[0]["depth"])
        censored = False
    else:
        tau = MAX_DEPTH + 1
        censored = True

    architecture = (
        group["architecture"].iloc[0]
        if "architecture" in group.columns
        else "UNKNOWN"
    )

    tau_records.append({
        "n": int(n),
        "k": int(k),
        "nk": int(n * k),
        "architecture": architecture,
        "tau_BP": tau,
        "censored": censored,
    })

tau_master = pd.DataFrame(tau_records)

tau_master = tau_master.sort_values(["n", "k"]).reset_index(drop=True)

print("=" * 80)
print("RECONSTRUCTED TAU_BP")
print("=" * 80)

display(tau_master)

print()
print(f"Total configurations: {len(tau_master)}")
print(
    f"Censored configurations: "
    f"{int(tau_master['censored'].sum())}"
)
print(
    f"Censoring rate: "
    f"{100 * tau_master['censored'].mean():.2f}%"
)

## Cell 6 — Establish the primary fitting domain

In [ ]:
# ============================================================
# CELL 6 — PRIMARY TRAINING DATASET
# ============================================================

train_df = tau_master[
    tau_master["n"].isin(TRAIN_N)
].copy()

heldout_df = tau_master[
    tau_master["n"].isin(HELD_OUT_N)
].copy()

if len(train_df) == 0:
    raise ValueError(
        "CRITICAL: No configurations found in the primary training domain."
    )

if len(heldout_df) == 0:
    print(
        "WARNING: No held-out configurations found for n=14/16."
    )

print("=" * 80)
print("PRIMARY DATA PARTITION")
print("=" * 80)

print(f"Training domain n: {TRAIN_N}")
print(f"Training configurations: {len(train_df)}")
print(
    f"Training censored: "
    f"{int(train_df['censored'].sum())}"
)

print()

for n_val in sorted(train_df["n"].unique()):
    ks = sorted(
        train_df.loc[
            train_df["n"] == n_val,
            "k"
        ].unique()
    )
    print(f"n={int(n_val)} -> k={ks}")

print()
print(f"Held-out n: {HELD_OUT_N}")
print(f"Held-out configurations: {len(heldout_df)}")

print("\nHeld-out configurations:")
display(
    heldout_df[
        ["n", "k", "nk", "tau_BP", "censored"]
    ]
)

## Cell 7 — Prepare log variables

In [ ]:
# ============================================================
# CELL 7 — LOG TRANSFORMS FOR MODEL FITTING
# ============================================================

# We make a fresh copy so raw tau_master remains untouched.
fit_df = train_df.copy()

fit_df["log_tau"] = np.log(fit_df["tau_BP"])
fit_df["log_n"] = np.log(fit_df["n"])
fit_df["log_k"] = np.log(fit_df["k"])
fit_df["log_nk"] = np.log(fit_df["nk"])
fit_df["log_k_over_n"] = np.log(fit_df["k"] / fit_df["n"])

print("Prepared log-transformed fitting variables:")
display(
    fit_df[
        ["n", "k", "tau_BP", "log_n", "log_k", "log_nk"]
    ].head()
)

## Cell 8 — Helper functions

In [ ]:
# ============================================================
# CELL 8 — MODEL EVALUATION HELPERS
# ============================================================

def calculate_metrics(y_true, y_pred, model):
    """
    Compute metrics on the original tau scale.
    """

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    residuals = y_true - y_pred

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # MAPE is defined here exactly as used in the previous analyses.
    mape = np.mean(
        np.abs(residuals / y_true)
    ) * 100

    r2 = r2_score(y_true, y_pred)

    k_params = len(model.params)
    n_obs = len(y_true)

    aic = float(model.aic)
    bic = float(model.bic)

    if n_obs > k_params + 1:
        aicc = (
            aic
            + (2 * k_params * (k_params + 1))
            / (n_obs - k_params - 1)
        )
    else:
        aicc = np.nan

    return {
        "R2_tau": float(r2),
        "MAE": float(mae),
        "RMSE": float(rmse),
        "MAPE_percent": float(mape),
        "AIC": aic,
        "AICc": float(aicc),
        "BIC": bic,
    }


def manual_loocv(formula, data):
    """
    Log-space OLS LOOCV.

    Returns predictions on the original tau scale.
    """

    preds = []
    actual = []

    data = data.reset_index(drop=True)

    for i in range(len(data)):

        train_sub = data.drop(index=i)
        test_sub = data.iloc[[i]]

        model = sm.OLS.from_formula(
            formula,
            data=train_sub
        ).fit()

        pred_log = float(model.predict(test_sub).iloc[0])
        pred_tau = float(np.exp(pred_log))

        actual_tau = float(test_sub["tau_BP"].iloc[0])

        preds.append(pred_tau)
        actual.append(actual_tau)

    preds = np.asarray(preds)
    actual = np.asarray(actual)

    residuals = actual - preds

    return {
        "LOOCV_MAE": float(np.mean(np.abs(residuals))),
        "LOOCV_RMSE": float(np.sqrt(np.mean(residuals ** 2))),
        "LOOCV_MAPE_percent": float(
            np.mean(np.abs(residuals / actual)) * 100
        ),
        "predictions": preds,
        "actual": actual,
    }

## Cell 9 — Fit the five candidate models

In [ ]:
# ============================================================
# CELL 9 — CANDIDATE MODEL FITTING
# ============================================================
#
# We deliberately do NOT assume the product model wins.
#
# Candidate models:
#
# 1. n only
# 2. k only
# 3. k/n
# 4. separate n and k exponents
# 5. common nk product exponent
#
# ============================================================

candidate_formulas = {
    "System-size only": "log_tau ~ log_n",

    "Support-size only": "log_tau ~ log_k",

    "Ratio-based": "log_tau ~ log_k_over_n",

    "Separate n,k exponents":
        "log_tau ~ log_n + log_k",

    "Product nk":
        "log_tau ~ log_nk",
}

model_results = []
model_objects = {}
prediction_table = fit_df[
    ["n", "k", "nk", "tau_BP"]
].copy()

for model_name, formula in candidate_formulas.items():

    model = sm.OLS.from_formula(
        formula,
        data=fit_df
    ).fit()

    model_objects[model_name] = model

    pred_log = model.predict(fit_df)
    pred_tau = np.exp(pred_log)

    metrics = calculate_metrics(
        fit_df["tau_BP"],
        pred_tau,
        model
    )

    loo = manual_loocv(
        formula,
        fit_df
    )

    row = {
        "Model": model_name,
        "Formula": formula,
        "Parameters": len(model.params),
        "R2_log": float(model.rsquared),
        **metrics,
        "LOOCV_MAE": loo["LOOCV_MAE"],
        "LOOCV_RMSE": loo["LOOCV_RMSE"],
        "LOOCV_MAPE_percent": loo["LOOCV_MAPE_percent"],
    }

    model_results.append(row)

    prediction_table[
        f"pred_{model_name}"
    ] = np.asarray(pred_tau)

model_comparison = pd.DataFrame(model_results)

print("=" * 80)
print("CANDIDATE MODEL COMPARISON")
print("=" * 80)

display(
    model_comparison[
        [
            "Model",
            "Parameters",
            "R2_tau",
            "MAE",
            "RMSE",
            "MAPE_percent",
            "AIC",
            "AICc",
            "BIC",
            "LOOCV_MAE",
            "LOOCV_RMSE",
            "LOOCV_MAPE_percent",
        ]
    ].sort_values("AICc")
)

## Cell 10 — Extract parameters for every model

In [ ]:
# ============================================================
# CELL 10 — MODEL PARAMETER TABLE
# ============================================================

parameter_rows = []

for model_name, model in model_objects.items():

    confidence = model.conf_int()

    for parameter_name in model.params.index:

        parameter_rows.append({
            "Model": model_name,
            "Parameter": parameter_name,
            "Coefficient": float(
                model.params[parameter_name]
            ),
            "Std_Error": float(
                model.bse[parameter_name]
            ),
            "t_statistic": float(
                model.tvalues[parameter_name]
            ),
            "p_value": float(
                model.pvalues[parameter_name]
            ),
            "CI_95_lower": float(
                confidence.loc[parameter_name, 0]
            ),
            "CI_95_upper": float(
                confidence.loc[parameter_name, 1]
            ),
        })

model_parameters = pd.DataFrame(parameter_rows)

display(model_parameters)

## Cell 11 — Explicitly recover the Product model

In [ ]:
# ============================================================
# CELL 11 — PRODUCT MODEL PARAMETERS
# ============================================================

product_model = model_objects["Product nk"]

log_A = float(
    product_model.params["Intercept"]
)

slope = float(
    product_model.params["log_nk"]
)

A_hat = float(np.exp(log_A))
c_hat = float(-slope)

c_se = float(
    product_model.bse["log_nk"]
)

c_ci_raw = product_model.conf_int().loc["log_nk"]

# Because c = -slope, reverse the confidence interval signs.
c_ci_low = float(-c_ci_raw[1])
c_ci_high = float(-c_ci_raw[0])

print("=" * 80)
print("PRODUCT MODEL — DATA-DERIVED PARAMETERS")
print("=" * 80)

print(f"A = {A_hat:.8f}")
print(f"c = {c_hat:.8f}")
print(f"SE(c) = {c_se:.8f}")
print(f"95% CI(c) = [{c_ci_low:.8f}, {c_ci_high:.8f}]")
print()
print(
    "Data-derived model:"
)
print(
    f"tau_BP = {A_hat:.8f} * (n*k)^(-{c_hat:.8f})"
)

## Cell 12 — Correct nested F-test

In [ ]:
# ============================================================
# CELL 12 — NESTED F-TEST
# ============================================================
#
# Restricted model:
#
#     log(tau) = log(A) - c*log(nk)
#
# Equivalent to:
#
#     log(tau) = log(A) - c*log(n) - c*log(k)
#
# Unrestricted model:
#
#     log(tau) = log(A) - a*log(n) - b*log(k)
#
# The F-test asks whether the additional independent exponent
# is statistically justified.
# ============================================================

restricted = model_objects["Product nk"]
unrestricted = model_objects["Separate n,k exponents"]

ssr_restricted = float(restricted.ssr)
ssr_unrestricted = float(unrestricted.ssr)

df_restricted = float(restricted.df_resid)
df_unrestricted = float(unrestricted.df_resid)

df_difference = df_restricted - df_unrestricted

if df_difference <= 0:
    raise RuntimeError(
        "CRITICAL: Nested model degrees of freedom are invalid."
    )

f_statistic = (
    (ssr_restricted - ssr_unrestricted) / df_difference
) / (
    ssr_unrestricted / df_unrestricted
)

f_pvalue = float(
    stats.f.sf(
        f_statistic,
        df_difference,
        df_unrestricted
    )
)

print("=" * 80)
print("NESTED F-TEST: PRODUCT vs SEPARATE n,k EXPONENTS")
print("=" * 80)

print(f"F-statistic: {f_statistic:.8f}")
print(f"p-value:     {f_pvalue:.8f}")
print(f"df1:         {df_difference:.0f}")
print(f"df2:         {df_unrestricted:.0f}")

if f_pvalue < 0.05:
    print(
        "\nRESULT: The unrestricted two-exponent model provides "
        "a statistically significant improvement at alpha=0.05."
    )
else:
    print(
        "\nRESULT: The additional independent exponent is NOT "
        "statistically justified at alpha=0.05."
    )

## Cell 13 — Determine the model ranking from the data

In [ ]:
# ============================================================
# CELL 13 — MODEL SELECTION SUMMARY
# ============================================================

ranking = model_comparison.copy()

# Lower is better for these metrics
ranking["AICc_rank"] = ranking["AICc"].rank(
    method="min",
    ascending=True
)

ranking["LOOCV_MAE_rank"] = ranking["LOOCV_MAE"].rank(
    method="min",
    ascending=True
)

ranking["RMSE_rank"] = ranking["RMSE"].rank(
    method="min",
    ascending=True
)

# Higher is better for R2
ranking["R2_rank"] = ranking["R2_tau"].rank(
    method="min",
    ascending=False
)

ranking = ranking.sort_values(
    ["AICc", "LOOCV_MAE"]
).reset_index(drop=True)

print("=" * 80)
print("DATA-DRIVEN MODEL RANKING")
print("=" * 80)

display(
    ranking[
        [
            "Model",
            "R2_tau",
            "AICc",
            "LOOCV_MAE",
            "LOOCV_RMSE",
            "MAE",
            "RMSE",
        ]
    ]
)

best_aicc_model = ranking.iloc[0]["Model"]
best_loocv_model = (
    model_comparison
    .sort_values("LOOCV_MAE")
    .iloc[0]["Model"]
)

print()
print(f"Best by AICc:       {best_aicc_model}")
print(f"Best by LOOCV MAE:  {best_loocv_model}")

## Cell 14 — Check the product-model prediction table

In [ ]:
# ============================================================
# CELL 14 — PRODUCT MODEL PREDICTIONS
# ============================================================

product_predictions = prediction_table[
    [
        "n",
        "k",
        "nk",
        "tau_BP",
        "pred_Product nk",
    ]
].copy()

product_predictions["residual"] = (
    product_predictions["tau_BP"]
    - product_predictions["pred_Product nk"]
)

product_predictions["absolute_error"] = np.abs(
    product_predictions["residual"]
)

product_predictions["percentage_error"] = (
    np.abs(
        product_predictions["residual"]
        / product_predictions["tau_BP"]
    )
    * 100
)

display(
    product_predictions.sort_values(["n", "k"])
)

## Cell 15 — Final baseline status

In [ ]:
# ============================================================
# CELL 15 — FINAL BASELINE STATUS
# ============================================================

product_row = model_comparison[
    model_comparison["Model"] == "Product nk"
].iloc[0]

print("\n" + "=" * 90)
print("OFFICIAL BASELINE RECONSTRUCTION SUMMARY")
print("=" * 90)

print(f"Raw dataset:             {INPUT_FILE}")
print(f"Training systems:        n={TRAIN_N}")
print(f"Held-out systems:        n={HELD_OUT_N}")
print(f"Threshold:               {BASELINE_THRESHOLD}")
print(f"Training configurations: {len(train_df)}")
print(
    f"Training censored:       "
    f"{int(train_df['censored'].sum())}"
)

print("\nProduct model:")
print(
    f"tau_BP = {A_hat:.8f} * (n*k)^(-{c_hat:.8f})"
)

print(f"A = {A_hat:.8f}")
print(
    f"c = {c_hat:.8f} "
    f"(95% CI [{c_ci_low:.8f}, {c_ci_high:.8f}])"
)

print(f"R² = {product_row['R2_tau']:.6f}")
print(f"MAE = {product_row['MAE']:.6f}")
print(f"RMSE = {product_row['RMSE']:.6f}")
print(f"MAPE = {product_row['MAPE_percent']:.6f}%")
print(f"LOOCV MAE = {product_row['LOOCV_MAE']:.6f}")

print("\nModel-selection information:")
print(f"Best AICc model:      {best_aicc_model}")
print(f"Best LOOCV-MAE model: {best_loocv_model}")
print(f"Nested F-test p:      {f_pvalue:.8f}")

print("\nIMPORTANT:")
print("- No fitted parameters were hardcoded.")
print("- tau_BP was derived directly from var_pooled.")
print("- n=14 and n=16 were excluded from fitting.")
print("- Censored configurations are explicitly identified.")

print("=" * 90)

## Cell 16 — Save all Notebook 1 deliverables

In [ ]:
# ============================================================
# CELL 16 — SAVE NOTEBOOK 1 DELIVERABLES
# ============================================================

# Baseline tau dataset used for fitting
train_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "baseline_tau_dataset.csv"
    ),
    index=False
)

# All reconstructed configurations
tau_master.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "all_reconstructed_tau.csv"
    ),
    index=False
)

# Candidate model comparison
model_comparison.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "baseline_model_comparison.csv"
    ),
    index=False
)

# Parameter table
model_parameters.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "baseline_model_parameters.csv"
    ),
    index=False
)

# Predictions
prediction_table.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "baseline_predictions.csv"
    ),
    index=False
)

# Product-specific prediction diagnostics
product_predictions.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "product_model_predictions.csv"
    ),
    index=False
)

# Depth audit
depth_audit_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "depth_coverage_audit.csv"
    ),
    index=False
)

# Held-out data
heldout_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "heldout_n14_n16.csv"
    ),
    index=False
)

# Machine-readable summary
summary = {
    "input_file": INPUT_FILE,
    "baseline_threshold": BASELINE_THRESHOLD,
    "training_n": TRAIN_N,
    "held_out_n": HELD_OUT_N,
    "training_configurations": int(len(train_df)),
    "training_censored": int(train_df["censored"].sum()),
    "product_A": A_hat,
    "product_c": c_hat,
    "product_c_standard_error": c_se,
    "product_c_CI95_low": c_ci_low,
    "product_c_CI95_high": c_ci_high,
    "product_R2": float(product_row["R2_tau"]),
    "product_MAE": float(product_row["MAE"]),
    "product_RMSE": float(product_row["RMSE"]),
    "product_MAPE_percent": float(product_row["MAPE_percent"]),
    "product_LOOCV_MAE": float(product_row["LOOCV_MAE"]),
    "best_AICc_model": best_aicc_model,
    "best_LOOCV_MAE_model": best_loocv_model,
    "nested_F_statistic": float(f_statistic),
    "nested_F_pvalue": f_pvalue,
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "baseline_summary.json"
    ),
    "w"
) as f:
    json.dump(summary, f, indent=2)

print("=" * 80)
print("NOTEBOOK 1 DELIVERABLES")
print("=" * 80)

for fname in sorted(os.listdir(OUTPUT_DIR)):
    print(
        os.path.join(
            OUTPUT_DIR,
            fname
        )
    )

print("\nSTATUS: COMPLETE")
print("Proceed to Notebook 2 only after reviewing these outputs.")
One critical note before you run it